# TEST INFERENCE PIPELINE

In [ ]:
from pathlib import Path
import numpy as np
import torch
import io
from PIL import Image
import cv2

from torch.utils.data import DataLoader
from emma import EmmaDataset, EmmaSearcher, SearchQuery
from emma.dataloader import EmmaMapDataset
from emma.dataloader import emma_collate

import utils.logging_ as logging

import matplotlib.pyplot as plt

In [ ]:
# Check CUDA Availability

if torch.cuda.is_available():
    print("Ready")
    device = torch.device('cuda')
else:
    print("Not Ready")
    device = torch.device('cpu')

## > DATASET

In [ ]:
dataset_root = None  # e.g. Path("/path/to/EMMa")

if dataset_root is None:
    raise ValueError("Please set `dataset_root` to the path of the EMMa dataset.")

dataset_root = Path(dataset_root)

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_root}")

In [ ]:
# INIT DATASET
ds = EmmaDataset(dataset_root)

In [ ]:
# QUERY
se = EmmaSearcher(ds)
query = SearchQuery(splits=["Test"], require_listed=["mass"], limit=10)
asins = se.query_asins(query)
print(len(asins))

In [ ]:
def my_transform(asin: str, rec: dict):
    image_dict = ds.get_images(rec)

    first_img_key, first_img_bytes = next(iter(image_dict.items()))
    first_img = Image.open(io.BytesIO(first_img_bytes)).convert("RGB")

    dimensions = np.squeeze(ds.get_package_dimensions(rec)) * 2.54 / 100.0
    normalization = np.float32(np.linalg.norm(dimensions))

    return {
        "image": np.array(first_img),
        "normalization": normalization,
        "asin": asin,
        "image_key": first_img_key,
    }

In [ ]:
loader = DataLoader(
    EmmaMapDataset(ds, asins, transform=my_transform),
    batch_size=8, shuffle=False, collate_fn=emma_collate,
)

## > INFERENCE

In [ ]:
def thresholding_segmentation(image_rgb, threshold, kernel_size, open_iterations):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    _, bin_img = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY_INV) # cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    bin_img = cv2.morphologyEx(bin_img, cv2.MORPH_OPEN, kernel, iterations=open_iterations)

    valid_mask = (bin_img > 127).astype(np.uint8, copy=False)

    return valid_mask

In [ ]:
masks = []
for batch_idx, batch in enumerate(loader):
    asin_batch = list(batch["asin"])
    image_batch = batch["image"]

    for asin, image in zip(asin_batch, image_batch, strict=True):
        mask = thresholding_segmentation(
            np.asarray(image),
            threshold=253,
            kernel_size=3,
            open_iterations=2,
        )
        masks.append(mask)

    logging.progress_bar(batch_idx, len(loader), 1, 1)
        
print("Done")

## > VISUALIZE

In [ ]:
for mask in masks:
    plt.imshow(mask, cmap='gray', interpolation='nearest')
    plt.axis('off')
    plt.show()